## Cell 1：检查全新 Kaggle 环境

In [1]:
import sys
import torch
import numpy as np
import sklearn
import yaml

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("PyTorch:", torch.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("没有检测到 GPU，请先在 Kaggle Settings 中开启 GPU。")

print("\nPASS: Kaggle 基础环境正常。")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Python executable: /usr/bin/python3
PyTorch: 2.10.0+cu128
NumPy: 2.0.2
scikit-learn: 1.6.1
CUDA available: True
GPU: Tesla T4

PASS: Kaggle 基础环境正常。


## Cell 2：检查 anndata 是否已经存在

In [2]:
import importlib.util

has_anndata = importlib.util.find_spec("anndata") is not None

print("anndata installed:", has_anndata)

anndata installed: False


## Cell 3：安装 anndata，不改动 NumPy

In [3]:
import sys
import subprocess

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-deps",
        "anndata==0.11.4",
    ],
    check=True,
)

print("PASS: anndata package installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 9.1 MB/s eta 0:00:00
PASS: anndata package installed.


## Cell 4：验证 anndata 是否可以正常导入

In [4]:
import numpy as np

print("NumPy before anndata import:", np.__version__)

import anndata

print("anndata:", anndata.__version__)
print("NumPy after import:", np.__version__)

print("\nPASS: anndata 可以正常导入。")

NumPy before anndata import: 2.0.2
anndata: 0.11.4
NumPy after import: 2.0.2

PASS: anndata 可以正常导入。


## Cell 5：重新克隆最新 SpaMGCL

In [5]:
from pathlib import Path
import shutil
import subprocess

repo_root = Path("/kaggle/working/SpaMGCL")
project_root = repo_root / "SpaMGCL"

if repo_root.exists():
    shutil.rmtree(repo_root)

subprocess.run(
    [
        "git",
        "clone",
        "--depth",
        "1",
        "--branch",
        "main",
        "https://github.com/huqian122/SpaMGCL.git",
        str(repo_root),
    ],
    check=True,
)

assert project_root.exists()

print("Repository root:")
print(repo_root)

print("\nProject root:")
print(project_root)

print("\nLatest commit:")
subprocess.run(
    ["git", "-C", str(repo_root), "log", "-1", "--oneline"],
    check=True,
)

Cloning into '/kaggle/working/SpaMGCL'...


Repository root:
/kaggle/working/SpaMGCL

Project root:
/kaggle/working/SpaMGCL/SpaMGCL

Latest commit:
b4abd3d Add files via upload


CompletedProcess(args=['git', '-C', '/kaggle/working/SpaMGCL', 'log', '-1', '--oneline'], returncode=0)

## Cell 6：检查 BSRR 代码与正式 YAML

In [6]:
from pathlib import Path
from pprint import pprint
import yaml
import os

project_root = Path("/kaggle/working/SpaMGCL/SpaMGCL")
os.chdir(project_root)

required_files = [
    "src/clustering/refinement.py",
    "src/clustering/predict.py",
    "experiments/run_exp.py",
    "scripts/audit_run.py",
    "tests/test_bsrr.py",
    "configs/final_clean/hlna1_clean_200.yaml",
    "configs/final_clean/d1_clean_200.yaml",
    "configs/final_clean/e185_clean_200.yaml",
    "configs/final_clean/e185_clean_smoke.yaml",
    "configs/final_clean/s2e15_clean_200.yaml",
    "configs/final_clean/s2e18_clean_200.yaml",
]

missing = []

for f in required_files:
    exists = Path(f).exists()
    print(f"{'OK' if exists else 'MISSING':8s} {f}")

    if not exists:
        missing.append(f)

assert not missing, f"缺少文件: {missing}"

with open(
    "configs/final_clean/e185_clean_smoke.yaml",
    "r",
    encoding="utf-8",
) as f:
    cfg = yaml.safe_load(f)

print("\n===== clustering =====")
pprint(cfg["clustering"])

print("\n===== refinement =====")
pprint(cfg["refinement"])

assert cfg["clustering"]["n_init"] == 20
assert cfg["clustering"]["random_state"] == 0

assert cfg["refinement"]["enabled"] is True
assert cfg["refinement"]["method"] == "bsrr"
assert cfg["refinement"]["spatial_k"] == 3

print("\nPASS: BSRR 代码和 YAML 正确。")

OK       src/clustering/refinement.py
OK       src/clustering/predict.py
OK       experiments/run_exp.py
OK       scripts/audit_run.py
OK       tests/test_bsrr.py
OK       configs/final_clean/hlna1_clean_200.yaml
OK       configs/final_clean/d1_clean_200.yaml
OK       configs/final_clean/e185_clean_200.yaml
OK       configs/final_clean/e185_clean_smoke.yaml
OK       configs/final_clean/s2e15_clean_200.yaml
OK       configs/final_clean/s2e18_clean_200.yaml

===== clustering =====
{'embedding': 'concat_z',
 'method': 'kmeans',
 'n_clusters': 14,
 'n_init': 20,
 'random_state': 0}

===== refinement =====
{'enabled': True, 'method': 'bsrr', 'spatial_k': 3}

PASS: BSRR 代码和 YAML 正确。


## Cell 7：检查 E18.5 数据路径

In [7]:
from pathlib import Path

data_root = Path("/kaggle/input/datasets/wuvdji/smgc-data")

print("Data root:", data_root)
print("Exists:", data_root.exists())

assert data_root.exists()

print("\nTop-level contents:")
for p in sorted(data_root.iterdir())[:30]:
    print(" -", p.name)

print("\nPASS: 数据路径正常。")

Data root: /kaggle/input/datasets/wuvdji/smgc-data
Exists: True

Top-level contents:
 - E18.5_mouse_brain
 - Human_Lymph_Nodes
 - Mouse_Embryos_S2
 - simulation

PASS: 数据路径正常。


## Cell 8：执行语法检查和 BSRR 单元测试

In [8]:
import subprocess
import sys
import os

os.chdir("/kaggle/working/SpaMGCL/SpaMGCL")

files_to_compile = [
    "src/clustering/refinement.py",
    "src/clustering/predict.py",
    "experiments/run_exp.py",
    "scripts/audit_run.py",
    "tests/test_bsrr.py",
]

for f in files_to_compile:
    subprocess.run(
        [sys.executable, "-m", "py_compile", f],
        check=True,
    )
    print("PASS:", f)

print("\n===== BSRR unittest =====")

subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        "tests",
        "-p",
        "test_bsrr.py",
        "-v",
    ],
    check=True,
)

print("\nPASS: syntax + BSRR tests 全部通过。")

PASS: src/clustering/refinement.py
PASS: src/clustering/predict.py
PASS: experiments/run_exp.py
PASS: scripts/audit_run.py
PASS: tests/test_bsrr.py

===== BSRR unittest =====


test_bsrr_rejects_spot_count_mismatch (test_bsrr.BSRRTest.test_bsrr_rejects_spot_count_mismatch) ... ok
test_bsrr_synthetic_contract (test_bsrr.BSRRTest.test_bsrr_synthetic_contract) ... ok
test_disabled_refinement_preserves_embedding (test_bsrr.BSRRTest.test_disabled_refinement_preserves_embedding) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.031s

OK



PASS: syntax + BSRR tests 全部通过。


## Cell 9：检查 sanity 输出目录

In [9]:
from pathlib import Path

output_dir = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/results_clean/e185_clean_smoke"
)

metrics_path = output_dir / "metrics.json"

print("Output directory:")
print(output_dir)

print("\nDirectory exists:", output_dir.exists())
print("metrics.json exists:", metrics_path.exists())

if metrics_path.exists():
    raise RuntimeError(
        "发现已有完整实验结果 metrics.json。"
        "为了避免覆盖旧实验，请先停止。"
    )

print("\nPASS: 可以开始新的 E18.5 sanity run。")

Output directory:
/kaggle/working/SpaMGCL/SpaMGCL/results_clean/e185_clean_smoke

Directory exists: False
metrics.json exists: False

PASS: 可以开始新的 E18.5 sanity run。


## Cell 10：运行 E18.5 50-epoch seed0 BSRR sanity experiment

In [10]:
import subprocess
import sys
import os

os.chdir("/kaggle/working/SpaMGCL/SpaMGCL")

cmd = [
    sys.executable,
    "experiments/run_exp.py",
    "--config",
    "configs/final_clean/e185_clean_smoke.yaml",
]

print("===== Experiment protocol =====")
print("Dataset              : E18.5")
print("Training seed        : 0")
print("Epochs               : 50")
print("Official embedding   : concat_z")
print("Refinement           : BSRR")
print("BSRR spatial_k       : 3")
print("KMeans n_init        : 20")
print("KMeans random_state  : 0")

print("\nRunning:")
print(" ".join(cmd))
print()

subprocess.run(cmd, check=True)

print("\nPASS: E18.5 50-epoch sanity training completed.")

===== Experiment protocol =====
Dataset              : E18.5
Training seed        : 0
Epochs               : 50
Official embedding   : concat_z
Refinement           : BSRR
BSRR spatial_k       : 3
KMeans n_init        : 20
KMeans random_state  : 0

Running:
/usr/bin/python3 experiments/run_exp.py --config configs/final_clean/e185_clean_smoke.yaml

Dataset: E18.5 | spots=2129 | device=cuda
Modalities: RNA, ATAC | label=Combined_Clusters
Spatial graph: shape=(2129, 2129), nnz=7784
Learning rates: backbone=0.001 | cluster_head=0.008 (8x)
epoch 001/050 | total=23.166977 | rec=0.160814 | mgcl=7.668721 | cluster=3.298035 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=8.967e-03 | neg_count=4530512 | snf_masked_positions=0 | effC=13.976 | gradC=0.000e+00
epoch 002/050 | total=23.126160 | rec=0.148333 | mgcl=7.659276 | cluster=3.297572 | spatial_loss=0.000000 | lambda_spatial=0 | sc_enabled=False | snf_enabled=False | mgcl_weight_std=7.602e-0

## Cell 11：运行完整 audit

In [11]:
import subprocess
import sys
import os

os.chdir("/kaggle/working/SpaMGCL/SpaMGCL")

cmd = [
    sys.executable,
    "scripts/audit_run.py",
    "results_clean/e185_clean_smoke",
]

print("Running audit:")
print(" ".join(cmd))
print()

subprocess.run(cmd, check=True)

Running audit:
/usr/bin/python3 scripts/audit_run.py results_clean/e185_clean_smoke

Dataset: E18.5
Seed: 0
Cluster warm-up epochs: 10
Cluster loss start epoch: 11
Configured lambda_cluster: 0.1
Official embedding: concat_z
Official clustering method: kmeans
KMeans n_init: 20
KMeans random_state: 0
Refinement enabled: True
Refinement method: bsrr
Refinement spatial_k: 3
Epochs: 50
lambda_rec: 1.0
lambda_mgcl: 3.0
lambda_cluster: 0.1
lambda_spatial: 0.0
ARI: 0.44989943491747353
NMI: 0.5360916872954723
NMI average method: max
recomputed ARI: 0.44989943491747353
recomputed NMI: 0.5360916872954723
Q argmax diagnostic ARI: 0.4989072074338289
Q argmax diagnostic NMI: 0.48899439592371513
z_concat shape: (2129, 512)
PASS


CompletedProcess(args=['/usr/bin/python3', 'scripts/audit_run.py', 'results_clean/e185_clean_smoke'], returncode=0)

## Cell 12：比较 raw concat-Z 与 BSRR official 结果

In [12]:
import json
import numpy as np

from pathlib import Path
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
)

output_dir = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/results_clean/e185_clean_smoke"
)

gt = np.load(output_dir / "gt_labels.npy")
pred_raw = np.load(output_dir / "pred_concat_z_kmeans.npy")
pred_bsrr = np.load(output_dir / "pred_labels.npy")

z_concat = np.load(output_dir / "z_concat.npy")
z_bsrr = np.load(output_dir / "z_bsrr.npy")

with (output_dir / "metrics.json").open("r", encoding="utf-8") as f:
    metrics = json.load(f)

raw_ari = adjusted_rand_score(gt, pred_raw)

raw_nmi = normalized_mutual_info_score(
    gt,
    pred_raw,
    average_method="max",
)

bsrr_ari = adjusted_rand_score(gt, pred_bsrr)

bsrr_nmi = normalized_mutual_info_score(
    gt,
    pred_bsrr,
    average_method="max",
)

print("===== Embeddings =====")
print("z_concat shape:", z_concat.shape)
print("z_bsrr shape :", z_bsrr.shape)
print(
    "Mean |BSRR - raw|:",
    float(np.mean(np.abs(z_bsrr - z_concat))),
)

print("\n===== Raw concat-Z + KMeans =====")
print(f"ARI = {raw_ari:.12f}")
print(f"NMI = {raw_nmi:.12f}")

print("\n===== BSRR + KMeans (official) =====")
print(f"ARI = {bsrr_ari:.12f}")
print(f"NMI = {bsrr_nmi:.12f}")

print("\n===== Change =====")
print(f"Delta ARI = {bsrr_ari - raw_ari:+.12f}")
print(f"Delta NMI = {bsrr_nmi - raw_nmi:+.12f}")

print("\n===== BSRR diagnostics =====")

for key in [
    "sigma_spatial",
    "sigma_latent",
    "confidence_mean",
    "confidence_std",
    "confidence_min",
    "confidence_max",
]:
    print(f"{key}: {metrics['refinement'][key]}")

===== Embeddings =====
z_concat shape: (2129, 512)
z_bsrr shape : (2129, 512)
Mean |BSRR - raw|: 0.004969606641680002

===== Raw concat-Z + KMeans =====
ARI = 0.438898262587
NMI = 0.524419640256

===== BSRR + KMeans (official) =====
ARI = 0.449899434917
NMI = 0.536091687295

===== Change =====
Delta ARI = +0.011001172330
Delta NMI = +0.011672047039

===== BSRR diagnostics =====
sigma_spatial: 1.0
sigma_latent: 0.3792871792253053
confidence_mean: 0.355430695338432
confidence_std: 0.09081429765981344
confidence_min: 0.06826654249394233
confidence_max: 0.593990496016743


## Cell 13：打包本次 E18.5 seed0 sanity 结果

In [13]:
import shutil
from pathlib import Path

source_dir = Path(
    "/kaggle/working/SpaMGCL/SpaMGCL/results_clean/e185_clean_smoke"
)

archive_base = Path(
    "/kaggle/working/e185_bsrr_smoke_seed0"
)

zip_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=source_dir,
)

zip_path = Path(zip_path)

print("Archive:", zip_path)
print("Exists:", zip_path.exists())
print(
    "Size (MB):",
    round(zip_path.stat().st_size / 1024 / 1024, 2),
)

assert zip_path.exists()

print("\nPASS: sanity result archived.")

Archive: /kaggle/working/e185_bsrr_smoke_seed0.zip
Exists: True
Size (MB): 20.08

PASS: sanity result archived.
